






## Webpage Extraction and Embedding (PolyU CUS)

### 1. Extracting raw text data

In [1]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader, SitemapLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
cus_URL = "https://www.polyu.edu.hk/cus/"
docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "html.parser")   # Strip the html syntax
    
    # Get the title of the page
    title = soup.title.string if soup.title else "No Title"

    # Attempt to find main content areas
    main_content = soup.find('main') or soup.find('article') or soup.find('div', class_='content')
    if main_content:
        text = re.sub(r"\n\n+", "\n\n", main_content.get_text()).strip()
    else:
        for element in soup.find_all(['header', 'footer', 'nav']):
            element.decompose()
        text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess newlines
    
    return f"{title}\n\n{text}"

cus_loader = RecursiveUrlLoader(
    max_depth=6,
    url=cus_URL,
    base_url=cus_URL,
    prevent_outside=True,
    exclude_dirs=[
        cus_URL+"about-ous",
        cus_URL+"about-cus",
        cus_URL+"Sitemap", 
        cus_URL+"sitemap",
        cus_URL+"Search-Result", 
        cus_URL+"search-result", 
        cus_URL+"internal",
        cus_URL+"-",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = cus_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    for key in unwanted_metadata:
        if key in doc.metadata:
            del doc.metadata[key]
    docs.append(doc)

https://www.polyu.edu.hk/cus/
https://www.polyu.edu.hk/cus/study/minor/
https://www.polyu.edu.hk/cus/study/minor/minor-in-research-and-innovation/
https://www.polyu.edu.hk/cus/non-local-gur-study/non-local-car-subjects/subject-on-offer/
https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/discipline-specific-requirements-major/
https://www.polyu.edu.hk/cus/staff/publications/
https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/academic-integrity/
https://www.polyu.edu.hk/cus/study/ug/
https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/academic-integrity/
https://www.polyu.edu.hk/cus/non-local-gur-study/non-local-car-subjects/non-local-car-subjects/
https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/publications/
https://www.

In [3]:
idx = 15
print(f"Extracted number of webpages in CUS: {len(docs)}")
print(docs[idx].page_content)
print(docs[idx].metadata.get('source'))

'''
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in CUS: 51
Curriculum Framework | College of Undergraduate Studies

Senior Year Intakes
Holders of Higher Diploma or Associate Degree may apply for senior year admission to our bachelor’s degree programmes. Senior Year curriculum for our Bachelor’s degree programmes for Associate Degree or Higher Diploma graduates normally take two years and require the completion of 60-75 credits. Those programmes that include professional training in industry, commerce and the public sector may take an additional year. The credit requirements for graduation vary by programme and are subject to annual review. Academic departments will advise students of the actual credit requirements after admission.
Similar to students admitted to articulation degree programmes, Senior Year intake students will follow a reduced GUR curriculum. The curriculum framework for Senior Year intakes is provided as follows:

Curriculum framework for Senior Year intakes from 2022/23:
Curriculum fra

"\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1) if re.search(r'/([^/]+)/?$', source) else "unknown"
    chunk.metadata["chunk_id"] = f"PolyU_CUS_{url_end}_chunk_{i}"

    chunk.page_content = f"--- PolyU CUS Website URL: {source} ---\n\n{chunk.page_content}"

print("Number of chunks: ", len(chunks))

Number of chunks:  115


In [5]:
print(chunks[20])

page_content='--- PolyU CUS Website URL: https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/publications/ ---

Publications | College of Undergraduate Studies

Guidance Notes - General University Requirements (For students admitted from 2022/23) (Released in Sept 2025)

Full version: Download (PDF)
Leaflet (For 4-Year Undergraduate students): Download (PDF)

Leaflet (For Senior Year Intakes and Articulation Programme Degree students): Download (PDF)

 
Guidance Notes - General University Requirements (For students admitted in 2021/22 or before) (Released in Sept 2025)

Full version: Download (PDF)

Leaflet: Download (PDF)

 
Academic Integrity (Updated in Sept 2025)

Leaflet (PDF)
Student Guide on Academic Integrity (PDF)' metadata={'source': 'https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/publications/', 'content_type': 'text/html; charset=utf-8', 'title': 'Publications | College of Undergraduate Studi

### 3. Document Embedding in Chroma

In [6]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [13]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

1218

In [8]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

In [9]:
for i, chunk in enumerate(chunks):
    print(f"Adding chunk {i+1}/{len(chunks)} to ChromaDB. Metadata: {chunk.metadata}\n")
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i + 3000)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Adding chunk 1/115 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | College of Undergraduate Studies', 'chunk_id': 'PolyU_CUS_cus_chunk_0'}

Adding chunk 2/115 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | College of Undergraduate Studies', 'chunk_id': 'PolyU_CUS_cus_chunk_1'}

Adding chunk 3/115 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/study/minor/', 'content_type': 'text/html; charset=utf-8', 'title': 'Minor Programmes | College of Undergraduate Studies', 'chunk_id': 'PolyU_CUS_minor_chunk_2'}

Adding chunk 4/115 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/cus/study/minor/minor-in-research-and-innovation/', 'content_type': 'text/html; charset=utf-8', 'title': 'Minor in Research and Innovation | College of Undergraduate Studies', 'description': 'The Minor in Research and Innovation aims t

### 4. Simple Testing

In [12]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What is the general university requirement for undergraduate student?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: --- PolyU CUS Website URL: https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/general-university-requirements/ ---

General University Requirements (GUR) | College of Undergraduate Studies

GUR for 4-Year Undergraduate Student 

 

Admitted in 2021/22 or before

Freshman Seminar

Language & Communication Requirements

Leadership & Intra-Personal Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

Admitted from 2022/23

Artificial Intelligence and Data Analytics Requirement

Innovation and Entrepreneurship Requirement

Language & Communication Requirements

Leadership Education and Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

For the details of curriculum framework of the General University Requirements (GUR), please click here....
Source: https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/general-university-requirements/
Chunk ID: PolyU_CUS_general-university-requirements_chunk_76

Con